# Magic Hour · InsightFace InSwapper Face Swap

Deterministic **local** face swap experiment — **not** Krea2 image editing.

| Input | Role |
| --- | --- |
| **Body** | Full-resolution scene — pose, expression, clothing, lighting, background |
| **Face** | Identity donor — ArcFace embedding transferred onto the selected body face |

**Architecture:** Detect (RetinaFace/SCRFD) → select face → ArcFace ID → InSwapper → soft blend → optional GFPGAN/CodeFormer → save.

**Why this vs Krea2:** InSwapper only replaces the selected face region. It does **not** regenerate the scene, so head size, pose, expression, eye direction, lighting, clothing, hairstyle, and every other person stay intact.

**Recommended:** edit §1 → **Runtime → Change runtime type → GPU (T4)** → **Runtime → Run all**.

| GPU | Typical warm time |
| --- | --- |
| **T4** (16 GB) | ~5–20 s / image |
| **A100** | ~2–8 s / image |

Cold start downloads ~500 MB of models (Drive-cached).


---
## 1 · Settings

Only these knobs are meant to change.


In [ ]:
# === User-facing knobs ===
ENGINE = "inswapper"          # pluggable swap engine (inswapper today; reface/ghost later)
RESTORE = "none"              # none | gfpgan | codeformer
RESTORE_FIDELITY = 0.5        # GFPGAN/CodeFormer face fidelity
BLEND_STRENGTH = 1.0
COLOR_MATCH = 0.25            # mild lighting match inside face mask
DEBUG = True                  # save intermediates under intermediates/

# Multi-person body: which face to swap
# largest | rightmost | leftmost | index (use BODY_FACE_INDEX)
BODY_FACE_POLICY = "largest"
BODY_FACE_INDEX = 0

# Identity image face selection
SOURCE_FACE_POLICY = "largest"
SOURCE_FACE_INDEX = 0

# Repo branch that contains src/headswap/inswap (not on main yet)
REPO_BRANCH = "ab/identity-scale-match-spp"
# Optional: pin exact headswap_V2 commit (None = branch tip)
PINNED_COMMIT = None

print("Settings")
print(f"  ENGINE={ENGINE}  RESTORE={RESTORE}  RESTORE_FIDELITY={RESTORE_FIDELITY}")
print(f"  BLEND_STRENGTH={BLEND_STRENGTH}  COLOR_MATCH={COLOR_MATCH}  DEBUG={DEBUG}")
print(f"  BODY_FACE_POLICY={BODY_FACE_POLICY}  BODY_FACE_INDEX={BODY_FACE_INDEX}")
print(f"  SOURCE_FACE_POLICY={SOURCE_FACE_POLICY}  SOURCE_FACE_INDEX={SOURCE_FACE_INDEX}")
print(f"  REPO_BRANCH={REPO_BRANCH}")
print(f"  PINNED_COMMIT={PINNED_COMMIT or '(none — use branch tip)'}")


---
## 1b · Force-sync repo _(run if you see `No module named headswap.inswap`)_

Colab often keeps a **stale notebook copy**. This cell repairs `/content/headswap_V2` in place.


In [ ]:
#@title Force-sync InSwapper branch + fix imports
from pathlib import Path
import os, sys, subprocess

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = str(globals().get("REPO_BRANCH") or "ab/identity-scale-match-spp")

if not Path("/content").exists():
    raise SystemExit("Colab only")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--prune"], check=False)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=False)
os.chdir(REPO)

inswap = REPO / "src" / "headswap" / "inswap"
assert inswap.is_dir(), f"still missing {inswap}"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=False)

src = str(REPO / "src")
if src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
for m in [k for k in list(sys.modules) if k == "headswap" or k.startswith("headswap.")]:
    del sys.modules[m]

import headswap.inswap  # noqa: F401
os.environ["HEADSWAP_REPO"] = str(REPO)
print("✓ headswap.inswap OK")
print("  branch:", subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip())
print("  commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip())
print("  path:", inswap)


---
## 2 · Setup _(first run / reconnect)_

One-click install for **T4**: InsightFace buffalo_l + InSwapper-128 (+ optional GFPGAN). No ComfyUI / Krea2.


In [ ]:
#@title Setup: GPU · Drive · InSwapper models
from pathlib import Path
import importlib.util
import os
import subprocess

assert Path("/content").exists(), "This notebook is for Google Colab (/content)."

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime → Change runtime type → GPU (T4 is enough), then Run all."
    )
print(f"✓ GPU  {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB)")

from google.colab import drive
print("→ Mounting Google Drive…")
drive.mount("/content/drive")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
# InSwapper lives on this branch (not merged to main yet)
branch = str(globals().get("REPO_BRANCH") or "ab/identity-scale-match-spp")
pin = globals().get("PINNED_COMMIT")

print(f"→ Syncing headswap_V2 @ {pin or branch}…")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--prune"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)

subprocess.run(["git", "fetch", "origin", branch], check=False)
subprocess.run(["git", "checkout", "-B", branch, f"origin/{branch}"], check=False)
subprocess.run(["git", "pull", "--ff-only", "origin", branch], check=False)

if pin:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", str(pin)], check=False)
    subprocess.run(["git", "checkout", str(pin)], check=False)

inswap_pkg = REPO / "src" / "headswap" / "inswap"
if not inswap_pkg.is_dir():
    raise SystemExit(
        f"Missing {inswap_pkg}. Wrong branch/commit. "
        f"Set REPO_BRANCH={branch!r} in §1, re-run §2 Setup."
    )
print(f"✓ Found inswap package at {inswap_pkg}")
print(f"  git: {subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()} "
      f"({subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()})")

!pip install -q -e .

spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

# Force-reload if an older install left a stale headswap without inswap
import sys
for mod in [m for m in list(sys.modules) if m == "headswap" or m.startswith("headswap.")]:
    del sys.modules[mod]
import headswap.inswap  # noqa: F401
print("✓ import headswap.inswap OK")

os.environ["HEADSWAP_REPO"] = str(REPO)
os.environ["INSWAP_CACHE"] = "/content/drive/MyDrive/headswap_inswap"
os.environ["INSWAP_LOCAL_CACHE"] = "/content/inswap_cache"

print("→ Running InSwapper bootstrap…")
!bash scripts/setup_inswapper_colab.sh
print("✓ Setup done.")


---
## 3 · Upload body & face


In [ ]:
import importlib.util
from pathlib import Path
from google.colab import files
from PIL import Image
from IPython.display import display, Markdown

UPLOAD_DIR = Path("/content/inswap_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

display(Markdown("### Upload **body** (scene / group photo)"))
uploaded_body = files.upload()
assert uploaded_body, "No body image uploaded"
body_name = next(iter(uploaded_body))
BODY_PATH = UPLOAD_DIR / "body.png"
Image.open(__import__("io").BytesIO(uploaded_body[body_name])).convert("RGB").save(BODY_PATH)

display(Markdown("### Upload **face** (identity donor)"))
uploaded_face = files.upload()
assert uploaded_face, "No face image uploaded"
face_name = next(iter(uploaded_face))
FACE_PATH = UPLOAD_DIR / "face.png"
Image.open(__import__("io").BytesIO(uploaded_face[face_name])).convert("RGB").save(FACE_PATH)

body_img = Image.open(BODY_PATH)
face_img = Image.open(FACE_PATH)
print(f"Body: {BODY_PATH}  {body_img.size}")
print(f"Face: {FACE_PATH}  {face_img.size}")
display(Markdown("**Body**"))
display(body_img.resize((min(512, body_img.width), int(body_img.height * min(512, body_img.width) / body_img.width))))
display(Markdown("**Face (identity)**"))
display(face_img.resize((min(256, face_img.width), int(face_img.height * min(256, face_img.width) / face_img.width))))


---
## 4 · Detect faces & choose who to swap

Shows all detections. Change `BODY_FACE_POLICY` / `BODY_FACE_INDEX` in §1 and re-run this cell if needed.


In [ ]:
#@title Detect faces (self-heals missing inswap)
import importlib.util, os, sys, subprocess
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image

REPO = Path(os.environ.get("HEADSWAP_REPO", "/content/headswap_V2"))
BRANCH = str(globals().get("REPO_BRANCH") or "ab/identity-scale-match-spp")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"

def _ensure_inswap():
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    if not (REPO / "src" / "headswap" / "inswap").is_dir():
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=False)
    src = str(REPO / "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    for m in [k for k in list(sys.modules) if k == "headswap" or k.startswith("headswap.")]:
        del sys.modules[m]
    os.environ["HEADSWAP_REPO"] = str(REPO)

_ensure_inswap()
try:
    from headswap.inswap.detect import InsightFaceDetector, select_face
    from headswap.inswap.viz import bgr_to_pil, draw_detections, pil_to_bgr
except ModuleNotFoundError:
    # FORCE HEAL
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all"], check=False)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=False)
    _ensure_inswap()
    from headswap.inswap.detect import InsightFaceDetector, select_face
    from headswap.inswap.viz import bgr_to_pil, draw_detections, pil_to_bgr

CACHE = Path(os.environ.get("INSWAP_LOCAL_CACHE", "/content/inswap_cache"))
CACHE.mkdir(parents=True, exist_ok=True)

detector = InsightFaceDetector()
detector.load(CACHE, device="cuda")

body_img = Image.open(BODY_PATH).convert("RGB")
body_bgr = pil_to_bgr(body_img)
faces = detector.detect(body_bgr)
print(f"Detected {len(faces)} face(s)")
for i, f in enumerate(faces):
    print(f"  #{i}  bbox={[round(x,1) for x in f.bbox]}  score={f.det_score:.3f}  area={f.area:.0f}")

selected = select_face(faces, policy=BODY_FACE_POLICY, index=BODY_FACE_INDEX)
print(f"Selected policy={BODY_FACE_POLICY} index={BODY_FACE_INDEX} → bbox={[round(x,1) for x in selected.bbox]}")

overlay = bgr_to_pil(draw_detections(body_bgr, faces, selected=selected))
display(Markdown("**Detections** (green = selected)"))
w = min(900, overlay.width)
display(overlay.resize((w, int(overlay.height * w / overlay.width))))


---
## 5 · Run face swap


In [ ]:
import importlib.util
import json, os, time
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image

REPO = Path(os.environ.get("HEADSWAP_REPO", "/content/headswap_V2"))
spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)
if not (REPO / "src" / "headswap" / "inswap").is_dir():
    raise SystemExit("headswap.inswap missing — re-run §2 Setup (checks out REPO_BRANCH).")

from headswap.inswap.pipeline import InSwapPipeline

CACHE = Path(os.environ.get("INSWAP_LOCAL_CACHE", "/content/inswap_cache"))
OUT_ROOT = Path("/content/inswap_outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUT_ROOT / f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

body_img = Image.open(BODY_PATH).convert("RGB")
face_img = Image.open(FACE_PATH).convert("RGB")

pipe = InSwapPipeline(
    cache_dir=CACHE,
    engine=ENGINE,
    restorer=RESTORE,
    device="cuda",
    blend_strength=BLEND_STRENGTH,
    color_match_strength=COLOR_MATCH,
    restore_fidelity=RESTORE_FIDELITY,
    reblend_after_engine=True,
)

print("→ Loading models…")
t_load = time.perf_counter()
pipe.load()
print(f"  load {time.perf_counter() - t_load:.1f}s")

print("→ Running pipeline…")
result = pipe.run(
    body_img,
    face_img,
    out_dir=RUN_DIR,
    face_policy=BODY_FACE_POLICY,
    face_index=BODY_FACE_INDEX,
    source_face_policy=SOURCE_FACE_POLICY,
    source_face_index=SOURCE_FACE_INDEX,
    save_intermediates=DEBUG,
)

# Symlink / copy latest
latest = OUT_ROOT / "LATEST_RESULT.png"
result.image.save(latest)
result.image.save(OUT_ROOT / "INSWAP_RESULT.png")

cfg = {
    "engine": ENGINE,
    "restorer": RESTORE,
    "restore_fidelity": RESTORE_FIDELITY,
    "blend_strength": BLEND_STRENGTH,
    "color_match": COLOR_MATCH,
    "body_face_policy": BODY_FACE_POLICY,
    "body_face_index": BODY_FACE_INDEX,
    "source_face_policy": SOURCE_FACE_POLICY,
    "source_face_index": SOURCE_FACE_INDEX,
    "body_size": list(body_img.size),
    "latency_s": result.latency_s,
    "meta": result.meta,
    "run_dir": str(RUN_DIR),
}
(RUN_DIR / "run_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")

print(f"✓ Done in {result.latency_s:.2f}s")
print(f"  faces={result.meta.get('faces_detected')}  engine={result.meta.get('engine')}")
print(f"  output: {RUN_DIR / 'result.png'}")
display(Markdown(f"**Result** · `{RUN_DIR}`"))
w = min(900, result.image.width)
display(result.image.resize((w, int(result.image.height * w / result.image.width))))


---
## 6 · Comparison — original · swapped · difference map

Difference map highlights **only** pixels that changed. In a correct run, change should be concentrated on the selected face.


In [ ]:
import importlib.util
import os
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image

REPO = Path(os.environ.get("HEADSWAP_REPO", "/content/headswap_V2"))
spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

from headswap.inswap.viz import difference_map, pil_to_bgr, bgr_to_pil, side_by_side

original = Image.open(BODY_PATH).convert("RGB")
swapped = Image.open(RUN_DIR / "result.png").convert("RGB")
# Match sizes if needed
if swapped.size != original.size:
    swapped = swapped.resize(original.size, Image.Resampling.LANCZOS)

orig_bgr = pil_to_bgr(original)
swap_bgr = pil_to_bgr(swapped)
diff_bgr = difference_map(orig_bgr, swap_bgr, amplify=4.0)
sbs_bgr = side_by_side(orig_bgr, swap_bgr, diff_bgr)

comp_path = RUN_DIR / "comparison_original_swapped_diff.png"
bgr_to_pil(sbs_bgr).save(comp_path)

display(Markdown("### Original | Swapped | Difference"))
comp = Image.open(comp_path)
w = min(1400, comp.width)
display(comp.resize((w, int(comp.height * w / comp.width))))

# Also show key intermediates when DEBUG
inter = RUN_DIR / "intermediates"
if inter.is_dir():
    display(Markdown("### Intermediates"))
    for name in [
        "01_detection.png",
        "02_aligned_face.png",
        "03_source_identity.png",
        "04_swapped_face.png",
        "07_blended.png",
        "08_final.png",
    ]:
        p = inter / name
        if p.is_file():
            im = Image.open(p)
            display(Markdown(f"**{name}**"))
            ww = min(480, im.width)
            display(im.resize((ww, max(1, int(im.height * ww / im.width)))))

print(f"Saved comparison → {comp_path}")


---
## 7 · Run summary


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown

cfg = json.loads((RUN_DIR / "run_config.json").read_text())
timing = (cfg.get("meta") or {}).get("timing_s") or {}
lines = [
    f"- **Run dir:** `{RUN_DIR}`",
    f"- **Engine:** `{cfg.get('engine')}`",
    f"- **Restorer:** `{cfg.get('restorer')}`",
    f"- **Faces detected:** {(cfg.get('meta') or {}).get('faces_detected')}",
    f"- **Latency:** {cfg.get('latency_s'):.2f}s",
    f"- **Body size:** {cfg.get('body_size')} (full resolution preserved)",
    f"- **Full scene regenerated:** {(cfg.get('meta') or {}).get('full_scene_regenerated')}",
]
if timing:
    lines.append("- **Stage timings:** " + ", ".join(f"{k}={v:.2f}s" for k, v in timing.items() if k != "total_s"))
display(Markdown("\n".join(lines)))
print("Latest result:", Path("/content/inswap_outputs/INSWAP_RESULT.png"))


---
## Notes

- **Krea2 is untouched.** This notebook uses `src/headswap/inswap/` only.
- **Group photos:** set `BODY_FACE_POLICY` / `BODY_FACE_INDEX` in §1; §4 previews the selection.
- **Swap engines:** `ENGINE = "inswapper"` today. New engines register in `headswap.inswap.engines.ENGINE_REGISTRY` without rewriting the pipeline.
- **Restoration:** `RESTORE = "gfpgan"` or `"codeformer"` enhances **only** the selected face region after the swap.
- **Models (Drive-cached):** `/content/drive/MyDrive/headswap_inswap/`
- **Outputs:** `/content/inswap_outputs/run_*/`

